# AIBL_DemRKL
Copied from the original project. Paths are relative to the main folder.
Original DemRKG generation and training settings are preserved.


In [ ]:
# Paths relative to the main folder; portable between Windows and Linux.
from pathlib import Path
import os
import sys
_starts = [Path.cwd(), Path(globals().get("__file__", Path.cwd() / "notebook")).resolve().parent]
CLINICALKG_ROOT = next(p for start in _starts for p in (start, *start.parents)
                      if (p / "full_primekg.py").is_file() and (p / "paths.py").is_file())
for _path in (CLINICALKG_ROOT.parent, CLINICALKG_ROOT):
    if str(_path) in sys.path:
        sys.path.remove(str(_path))
    sys.path.insert(0, str(_path))
from clinicalKG.paths import PROJECT_ROOT, SHARED_PRIMEKG, raw_primekg_path, work_dir, asset_path
DATASET = 'AIBL'
METHOD = 'DemRKL'
WORK_DIR = work_dir(DATASET, METHOD)
DATASET_DIR = PROJECT_ROOT / DATASET
if METHOD == "PrimeKGL":
    SHARED_PRIMEKG.mkdir(parents=True, exist_ok=True)
# A notebook kernel may previously have imported the original project's modules.
for _name in list(sys.modules):
    if _name == "experiments" or _name.startswith("experiments."):
        _file = getattr(sys.modules[_name], "__file__", None)
        if _file and not Path(_file).resolve().is_relative_to(CLINICALKG_ROOT):
            del sys.modules[_name]
if "model1" in sys.modules:
    _file = getattr(sys.modules["model1"], "__file__", None)
    if _file and Path(_file).resolve().parent != WORK_DIR:
        del sys.modules["model1"]
if str(WORK_DIR) in sys.path:
    sys.path.remove(str(WORK_DIR))
sys.path.insert(0, str(WORK_DIR))
if DATASET_DIR.is_dir():
    os.chdir(DATASET_DIR)
os.environ.setdefault("KG_ABLATION_OUTPUT", str(CLINICALKG_ROOT / "results"))


In [ ]:
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# ================= 配置区 =================
RAW_KG_PATH = str(raw_primekg_path())
OUTPUT_CSV_PATH = str(WORK_DIR / 'AIBL_Seed_DementiaHKG.csv')

def build_seed_and_export_csv():
    print(f"==================================================")
    print(f"🕵️ 步骤 1: 加载 PrimeKG 数据库...")
    try:
        df = pd.read_csv(RAW_KG_PATH, low_memory=False)
        all_nodes = set(df['x_name'].dropna()).union(set(df['y_name'].dropna()))
        print(f"   ✅ 成功加载 {RAW_KG_PATH}，共提取独立节点 {len(all_nodes)} 个。")
    except FileNotFoundError:
        print(f"   ❌ 找不到 {RAW_KG_PATH}，请检查路径。")
        return

    print(f"\n==================================================")
    print(f"🧬 步骤 2: 初始化医学共识核心种子 (Core Seeds)...")
    
    # 【核心升级】引入 AIBL 关注的前沿血液/脑脊液生物标志物及经典基因
    core_seeds = {
        "Alzheimer disease": "Core_Disease", 
        "Dementia": "Core_Disease",
        "Cognitive impairment": "Core_Disease",
        "APOE": "Core_Gene",
        "MAPT": "Core_Gene",     # Tau蛋白
        "APP": "Core_Gene",      # 淀粉样蛋白前体
        "PSEN1": "Core_Gene",
        "PSEN2": "Core_Gene",
        "TREM2": "Core_Gene",
        "GFAP": "Core_Protein",  # 胶质纤维酸性蛋白 (AIBL重点)
        "NEFL": "Core_Protein"   # 神经丝轻链 NfL (AIBL重点)
    }
    print(f"   定义了 {len(core_seeds)} 个核心医学种子（已注入 GFAP, NEFL, MAPT 等前沿标志物）。")

    print(f"\n==================================================")
    print(f"📊 步骤 3: 确立 AIBL 临床与表型映射字典...")
    
    # 【精准瘦身】仅保留 AIBL 数据集中存在的有效客观特征
    aibl_mapping = {
        # --- 认知与功能量表 ---
        "mmse": "Cognitive impairment",  # 简易精神状态检查
        "cdr": "Dementia",               # 临床痴呆评定
        
        # --- 深度神经心理学记忆量表 ---
        "lm_imm": "Memory impairment",   # 逻辑记忆即刻回忆
        "lm_del": "Memory impairment",   # 逻辑记忆延迟回忆
        
        # --- 基因型特征 ---
        "apoe": "APOE"                   # APOE基因型
    }
    print(f"   定义了 {len(aibl_mapping)} 个 AIBL 特征到 PrimeKG 节点的映射规则。")

    print(f"\n==================================================")
    print(f"🔍 步骤 4: 在 PrimeKG 中校验节点有效性...")
    
    valid_records = []
    
    # 校验核心种子
    print("   [校验核心种子]")
    for seed, category in core_seeds.items():
        if seed in all_nodes:
            valid_records.append({
                "AIBL_Feature": "Core_Prior_Knowledge",
                "PrimeKG_Node": seed,
                "Category": category
            })
            print(f"     ✅ 命中: {seed}")
        else:
            print(f"     ❌ 未命中: {seed} (将被丢弃)")

    # 校验 AIBL 映射
    print("   [校验 AIBL 映射特征]")
    for aibl_feat, pkg_node in aibl_mapping.items():
        if pkg_node in all_nodes:
            valid_records.append({
                "AIBL_Feature": aibl_feat,
                "PrimeKG_Node": pkg_node,
                "Category": "Clinical_Mapping"
            })
            print(f"     ✅ 命中: {aibl_feat} -> {pkg_node}")
        else:
            print(f"     ❌ 未命中: {aibl_feat} -> {pkg_node} (请检查 PrimeKG 命名)")

    print(f"\n==================================================")
    print(f"🚫 步骤 5: 附加全局黑名单规则...")
    
    # 保留防止药物网络干扰的黑名单
    valid_records.append({"AIBL_Feature": "BLACKLIST_TYPE", "PrimeKG_Node": "drug", "Category": "Rule"})
    valid_records.append({"AIBL_Feature": "BLACKLIST_TYPE", "PrimeKG_Node": "exposure", "Category": "Rule"})
    valid_records.append({"AIBL_Feature": "BLACKLIST_REL", "PrimeKG_Node": "indication", "Category": "Rule"})
    valid_records.append({"AIBL_Feature": "BLACKLIST_REL", "PrimeKG_Node": "contraindication", "Category": "Rule"})
    valid_records.append({"AIBL_Feature": "BLACKLIST_REL", "PrimeKG_Node": "off-label use", "Category": "Rule"})
    print("   已将药物节点(drug)及用药关系等 5 条黑名单规则编入导出列表。")

    print(f"\n==================================================")
    print(f"💾 步骤 6: 导出为 CSV 文件...")
    
    out_df = pd.DataFrame(valid_records)
    out_df.to_csv(OUTPUT_CSV_PATH, index=False, encoding='utf-8')
    
    print(f"   ✅ 导出完成！总计保留 {len(out_df)} 行有效配置。")
    print(f"   📄 文件已保存至: {OUTPUT_CSV_PATH}")
    print(f"==================================================")

if __name__ == "__main__":
    build_seed_and_export_csv()